# 💸 Notebook 1: The Double-Charge Bug

**Goal:** see *why* naive APIs double-charge customers on retries.

Networks are unreliable. A client sends `POST /charge`. The server processes it and sends a `200 OK`. The response packet is lost. The client sees a timeout and retries. The server — with no memory of the first call — charges again.

Most payment / messaging / ordering bugs trace back to this pattern.

## 🧭 Quick primer: which HTTP methods are idempotent?

An operation is **idempotent** if calling it once has the same effect as calling it N times.

| Method   | Idempotent by spec? | Why |
|---------:|:-------------------:|-----|
| `GET`    | ✅ | just reads data |
| `PUT`    | ✅ | replaces the whole resource — doing it twice leaves the same state |
| `DELETE` | ✅ | deleting something that's already gone is a no-op |
| `POST`   | ❌ | each call usually creates a *new* resource / side-effect |

`POST /charge` is the dangerous one. It's our subject for the rest of this lab.

## 🛠️ Setup

```bash
cd 04-patterns/idempotency
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 🟥 BAD: server keeps no memory of requests

We'll simulate a flaky network: the server's response is lost some fraction of the time, so the client retries.

In [ ]:
import random
random.seed(1)

balances = {'alice': 100}

def server_charge(account, amount):
    # side-effect: money moves
    balances[account] -= amount
    return {'ok': True, 'balance': balances[account]}

def flaky_network_call(account, amount, drop_response_prob=0.6):
    """Server ALWAYS processes the request. But the response is sometimes lost."""
    result = server_charge(account, amount)
    if random.random() < drop_response_prob:
        raise TimeoutError('response lost in the network')
    return result

def client_charge_with_retry(account, amount, max_attempts=5):
    for attempt in range(1, max_attempts + 1):
        try:
            r = flaky_network_call(account, amount)
            print(f'  attempt {attempt}: ✅ got response {r}')
            return r
        except TimeoutError:
            print(f'  attempt {attempt}: ⏱️ timeout — retrying')
    raise RuntimeError('gave up')

print('Client asked to charge $10 once:')
client_charge_with_retry('alice', 10)
print(f"\nAlice's balance ended at {balances['alice']} — the client intended ONE charge of $10.")
print(f"Every silent retry became a real charge. 😱")


### 🔍 What went wrong?

- The *request* arrived at the server successfully every time.
- Only the *response* was lost.
- The server has no way to tell "this is a retry of the previous request" apart from "this is a brand-new charge".

Notice the client can't fix this on its own. From the client's side, "the response was lost" and "the server never got it" look **identical** — so it must retry, and retrying is what causes the damage. The fix has to live on the server.

## 🧭 Natural idempotency vs key-based idempotency

The table at the top says `PUT` is idempotent and `POST` isn't. That's a real, useful distinction — and it's worth seeing it run, because it tells you when you need an idempotency key **at all**.

- **Naturally idempotent**: the operation *states the desired end state* (`set balance to 90`, `set status to shipped`). Repeat it and nothing changes, because the second call asks for a state that already holds. **No key needed.**
- **Not naturally idempotent**: the operation states a *delta* (`subtract 10`, `append a row`, `increment`). Repeat it and you get a second delta. **This is where you need a key.**

Note what this is *not* about: it isn't the HTTP verb. A `PUT` that appends to a list is not idempotent, and a `POST` whose body carries the full target state effectively is. The verb is a convention; the **shape of the operation** is the actual property.

In [ ]:
state = {'alice': {'balance': 100, 'status': 'pending'}}

def put_set_status(account, status):
    """Naturally idempotent: names the END STATE. Running it twice is a no-op."""
    state[account]['status'] = status
    return state[account]['status']

def post_subtract(account, amount):
    """Not idempotent: names a DELTA. Running it twice subtracts twice."""
    state[account]['balance'] -= amount
    return state[account]['balance']

print('PUT  status=shipped  x3 ->', [put_set_status('alice', 'shipped') for _ in range(3)])
print('POST subtract 10     x3 ->', [post_subtract('alice', 10) for _ in range(3)])
print()
print('The PUT landed on the same value every time — retry it freely, no key needed.')
print('The POST drifted 100 -> 70. THAT is the operation that needs an idempotency key.')

### 🛠️ Can you just make everything naturally idempotent?

Sometimes, and it's the cheapest fix when it works:

| Delta operation | Naturally idempotent rewrite |
|---|---|
| `balance -= 10` | `UPDATE … SET balance = 90 WHERE balance = 100` (compare-and-set) |
| `INSERT INTO orders …` | `INSERT … ON CONFLICT (order_id) DO NOTHING` with a client-supplied id |
| `status = next(status)` | `SET status = 'shipped' WHERE status = 'packed'` |
| `send_email()` | can't — the side effect is outside your database |

The last row is the catch, and it's why the rest of this lab exists. Once a side effect leaves your database — a card charge, an email, a webhook, a message on a queue — you can't express it as an end state, so you need a **key** plus a **record that it happened**. That's notebooks 2-4.

👉 Next notebook: the client attaches an **idempotency key** so the server can tell a retry from a new request.